In [2]:
import math
import sys
from pathlib import Path

_here = Path().resolve()
for _p in [_here, *_here.parents]:
    _src = _p / "src"
    if (_src / "qudits_on_qubits" / "__init__.py").is_file():
        repo_root = _p
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
else:
    raise ImportError(
        "qudits_on_qubits repo root not found; run the notebook from notebooks/ or the repo root"
    )

from qiskit.quantum_info import Statevector, Operator, partial_trace, SparsePauliOp
from qiskit import qpy, QuantumCircuit
import numpy as np
from qiskit.synthesis import TwoQubitWeylDecomposition
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, Batch

from qudits_on_qubits import create_ame_circuit, generate_b_ame
from sympy.functions.combinatorial.numbers import legendre_symbol
from IPython.display import display, Math
from itertools import product
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from igraph import Graph, plot
import matplotlib.pyplot as plt

In [3]:
from qudits_on_qubits.bell_measurements.sampler_circuits import build_sampler_circuits_from_graph, build_sampler_circuits_for_candidate
from iqm.iqm_client import CircuitCompilationOptions, DDMode, STANDARD_DD_STRATEGY, DDStrategy
from iqm.qiskit_iqm import IQMProvider
from iqm.qiskit_iqm.iqm_naive_move_pass import transpile_to_IQM
from iqm.iqm_client.transpile import ExistingMoveHandlingOptions

from qudits_on_qubits.bell_measurements.sampler_circuits import run_sampler_circuits_to_counts_by_setting
from qudits_on_qubits.bell_measurements.sampler_circuits import decoding_kwargs_from_metadata
from qudits_on_qubits.bell_measurements.postprocessing import compute_bell_value_from_counts

In [4]:
dd_options = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=STANDARD_DD_STRATEGY,
  )

dd_default = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED)

dd_xy4 = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=DDStrategy(
          gate_sequences=[(5, "YXYX", "asap")]
      ),
  )

dd_xy8 = CircuitCompilationOptions(dd_mode=DDMode.ENABLED, 
                                   dd_strategy=DDStrategy(
                                       gate_sequences=[(9, 'XYXYYXYX', 'center')]))

In [50]:
import os
from dotenv import load_dotenv

In [ ]:
assert os.getenv("IQM_TOKEN"), "IQM_TOKEN is required"

In [ ]:
load_dotenv()
assert os.getenv("IQM_TOKEN"), "IQM_TOKEN is required"

In [54]:
provider_garnet = IQMProvider("https://resonance.iqm.tech/", quantum_computer="garnet")
backend_garnet = provider_garnet.get_backend(use_metrics=True)

In [5]:
from provider import get_backend_error_profile, to_static_architecture
from iqm.qiskit_iqm.fake_backends.iqm_fake_backend import IQMFakeBackend

In [ ]:
error_profile = get_backend_error_profile(backend=backend_garnet)

static_garnet_architecture = to_static_architecture(backend_garnet.architecture)

garnet_noisy_backend = IQMFakeBackend(architecture=static_garnet_architecture, error_profile=error_profile)

In [ ]:
selected_candidate = "monomial_full__sup012_P021_ph012"
#QuditsOnQubits\artifacts\iqm_runs\selected_best\two_qutrit\stage2_top10_rerun20_20260706\exact\rank01_monomial_full__sup023_P012_ph022
artifact_circuit_dir = repo_root / "artifacts" / "iqm_runs" / "raw" / "quantum_circuits" / "garnet" / "ghz3_qpy13" / selected_candidate
legacy_circuit_dir = repo_root.parent / "QuditsOnQubits" / "basis_direct_encoding_benchmarks" / "quantum_circuits" / "ghz3_qpy13" / selected_candidate

for circuit_dir in (artifact_circuit_dir, legacy_circuit_dir):
    if (circuit_dir / "graph_state_direct_basis.qpy").is_file():
        break
else:
    raise FileNotFoundError(
        "Circuit artifacts not found. Checked:\n"
        f"- {artifact_circuit_dir}\n"
        f"- {legacy_circuit_dir}"
    )

print(f"Loading circuits from: {circuit_dir}")

with (circuit_dir / "graph_state_direct_basis.qpy").open("rb") as f:
    testqc = qpy.load(f)[0]

with (circuit_dir / "graph_state_direct_basis_transpiled.qpy").open("rb") as f:
    qcsuptrans = qpy.load(f)[0]

with (circuit_dir / "F3_W.qpy").open("rb") as f:
    F3sup = qpy.load(f)[0]

Esup = np.load(circuit_dir / "E.npy")

In [ ]:
qcsuptrans.draw("mpl")

In [ ]:
qcsuptrans.depth()

In [ ]:
def fold_cz_preserve_layout(qc: QuantumCircuit, n: int = 3) -> QuantumCircuit:
      if n < 1:
          raise ValueError("n musi byc >= 1")
      if n % 2 == 0:
          raise ValueError("Do ZNE uzyj nieparzystego n: 1, 3, 5, ...")

      out = qc.copy()
      old_data = list(out.data)
      out.data.clear()

      for inst in old_data:
          op = inst.operation
          qargs = inst.qubits
          cargs = inst.clbits

          if op.name.lower() == "cz":
              for _ in range(n):
                  out.append(op.copy(), qargs, cargs)
          else:
              out.append(op.copy(), qargs, cargs)

      out.name = f"{qc.name}_czfold{n}"
      return out

In [ ]:
layout = qcsuptrans.layout.final_index_layout(filter_ancillas=True)
qutrit_qubits = ((layout[0], layout[1]), (layout[2], layout[3]), (layout[4], layout[5]))

sampler_circuits, metadata = build_sampler_circuits_for_candidate(candidate="ghz3", state_circuit=qcsuptrans, E=Esup, qutrit_qubits=qutrit_qubits)
isa_sampler_qc = [transpile_to_IQM(qc, backend=backend_garnet, optimization_level=3, seed_transpiler=9) for qc in sampler_circuits]

In [ ]:
isa_sampler_qc[0].draw("mpl", idle_wires=False, fold=50)

In [ ]:
isa_sampler_qc[0].depth()

In [ ]:
isa_sampler_qc_1 = isa_sampler_qc
isa_sampler_qc_3 = [fold_cz_preserve_layout(qc, n=3) for qc in isa_sampler_qc]
isa_sampler_qc_5 = [fold_cz_preserve_layout(qc, n=5) for qc in isa_sampler_qc]

In [ ]:
counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=AerSampler())
counts_by_setting3, run_info3 = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc_3, metadata, shots=1024*20, transpile_circuits=False, backend=AerSampler())
counts_by_setting5, run_info5 = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc_5, metadata, shots=1024*20, transpile_circuits=False, backend=AerSampler())

In [ ]:
bell_value = compute_bell_value_from_counts(counts_by_setting, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value3 = compute_bell_value_from_counts(counts_by_setting3, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value5 = compute_bell_value_from_counts(counts_by_setting5, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

bell_value, bell_value3, bell_value5

In [ ]:
metadata["terms"]

GHZ DD DEFAULT

In [ ]:
# counts_by_setting_garnet, run_info_garnet = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_default})


In [ ]:
# bell_value = compute_bell_value_from_counts(counts_by_setting_garnet, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [ ]:
# bell_value

GHZ DD XY4

In [ ]:
# counts_by_setting_garnet, run_info_garnet = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_xy4})
# bell_value_xy4 = compute_bell_value_from_counts(counts_by_setting_garnet, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
# bell_value_xy4

GHZ DD XY8

In [ ]:
counts_by_setting_garnet_xy8, run_info_garnet_xy8 = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_xy8})

In [ ]:
bell_value_xy8 = compute_bell_value_from_counts(counts_by_setting_garnet_xy8, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [ ]:
bell_value_xy8

In [ ]:
counts_by_setting_garnet3_xy8, run_info_garnet3_xy8 = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc_3, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_xy8})

In [ ]:
bell_value3_xy8 = compute_bell_value_from_counts(counts_by_setting_garnet3_xy8, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [ ]:
bell_value3_xy8

In [ ]:
counts_by_setting_garnet5_xy8, run_info_garnet5_xy8 = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc_5, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_xy8})

In [ ]:
bell_value5_xy8 = compute_bell_value_from_counts(counts_by_setting_garnet5_xy8, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [ ]:
bell_value5_xy8

ZNE with DD XY8 without measurement miti

In [ ]:
scale_factors = np.array([1.0, 3.0, 5.0])
bell_values = np.array([bell_value_xy8.real, bell_value3_xy8.real, bell_value5_xy8.real], dtype=float)

coeff = np.polyfit(scale_factors, bell_values, deg=1)
zne_linear = np.polyval(coeff, 0.0)

In [ ]:
zne_linear

MEAS miti

In [ ]:
import mthree

In [ ]:
mapping = mthree.utils.final_measurement_mapping(isa_sampler_qc[0])

In [ ]:
mapping

In [ ]:
measured_physical_qubits = sorted(set(mapping.values()))
measured_physical_qubits

In [ ]:
from qiskit.circuit import QuantumCircuit
import numpy as np

def build_readout_calibration_matrices(
    backend,
    physical_qubits,
    shots=10_000
):
    """
    Zwraca listę macierzy kalibracyjnych M3.
    Dla kubitów, których nie kalibrujemy, pozostaje None.
    """
    matrices = [None] * backend.num_qubits

    for q in physical_qubits:
        # Przygotowanie |0> i pomiar kubitu q
        cal_0 = QuantumCircuit(backend.num_qubits, 1)
        cal_0.measure(q, 0)

        # Przygotowanie |1> i pomiar kubitu q
        cal_1 = QuantumCircuit(backend.num_qubits, 1)
        cal_1.x(q)
        cal_1.measure(q, 0)

        counts_0, counts_1 = backend.run(
            [cal_0, cal_1],
            shots=shots
        ).result().get_counts()

        # P(odczytano 1 | przygotowano 0)
        p10 = counts_0.get("1", 0) / shots

        # P(odczytano 0 | przygotowano 1)
        p01 = counts_1.get("0", 0) / shots

        # Kolumny: stan przygotowany |0>, |1>
        # Wiersze: stan odczytany  |0>, |1>
        matrices[q] = np.array(
            [
                [1 - p10, p01],
                [p10, 1 - p01]
            ],
            dtype=np.float32
        )

        print(f"Qubit {q}:")
        print(matrices[q])

    return matrices

In [ ]:
calibration_matrix = [
    np.array([
        [0.9944, 0.0811],
        [0.0056, 0.9189]
    ]),
    np.array([
        [0.9965, 0.0117],
        [0.0035, 0.9883]
    ]),
    np.array([
        [0.9962, 0.0247],
        [0.0038, 0.9753]
    ]),
    np.array([
        [0.991,  0.0204],
        [0.009,  0.9796]
    ])
]

In [ ]:
measured_physical_qubits

In [ ]:
calibration_matrix

In [ ]:
calibration_matrix = build_readout_calibration_matrices(backend_garnet, measured_physical_qubits, shots=10000)

In [ ]:
mit = mthree.M3Mitigation()

mit.cals_from_matrices(calibration_matrix)

def apply_mitigation(counts_by_setting, mit, mapping):
    quasi = []

    for res in list(counts_by_setting.values()):
        quasi.append(mit.apply_correction(res, mapping, return_mitigation_overhead=True))

    quasi_miti = {}

    for i, setting in zip(quasi, list(counts_by_setting.keys())):
        quasi_temp = {}
        for key in i.keys():
            quasi_temp[key] = int(i[key]*1024*20)
        quasi_miti[setting] = quasi_temp

    return quasi_miti

In [ ]:
counts_by_setting_garnet_miti = apply_mitigation(counts_by_setting_garnet_xy8, mit, mapping)
counts_by_setting_garnet_miti3 = apply_mitigation(counts_by_setting_garnet3_xy8, mit, mapping)
counts_by_setting_garnet_miti5 = apply_mitigation(counts_by_setting_garnet5_xy8, mit, mapping)

In [ ]:
bell_value_miti = compute_bell_value_from_counts(counts_by_setting_garnet_miti, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value_miti3 = compute_bell_value_from_counts(counts_by_setting_garnet_miti3, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value_miti5 = compute_bell_value_from_counts(counts_by_setting_garnet_miti5, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [ ]:
scale_factors = np.array([1.0, 3.0, 5.0])
bell_values = np.array([bell_value_miti.real, bell_value_miti3.real, bell_value_miti5.real], dtype=float)

coeff = np.polyfit(scale_factors, bell_values, deg=1)
zne_linear = np.polyval(coeff, 0.0)

In [ ]:
zne_linear

In [ ]:
zne_linear

In [ ]:
6 * math.cos(math.pi / 9)